# PaddleOCR-VL: PDF → paddle.json + paddle_blocks.json

Reads one paper with PaddleOCR-VL and writes, to `thesis/papers/<PAPER_ID>/` on Drive:
- `paddle.json`: every display formula as LaTeX, with its equation number and page
- `paddle_blocks.json`: every block on every page in reading order (text, titles, tables as HTML)

Copy both files into `extractor_agent/papers/<PAPER_ID>/` and run the extractor locally.

**Runtime:** GPU (T4).

**Next paper in the same session:** change `PAPER_ID` in Settings, upload its PDF, then run A2 to A5 again (no reinstall, no restart). A2 clears the previous paper's pages.


## 1. Settings

In [ ]:
PAPER_ID = 'cheema_2018'                                   # one folder per paper
THESIS   = '/content/drive/MyDrive/thesis'
PDF      = f'{THESIS}/papers/{PAPER_ID}.pdf'               # upload the PDF here first
OUT_DIR  = f'{THESIS}/papers/{PAPER_ID}'

PAGES    = None     # 0-indexed pages to read; None = all pages (needed for the text and tables)
DPI      = 150      # VLMs are trained on ~1-2k px images; 300 DPI costs memory for no gain


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(OUT_DIR, exist_ok=True)   # after mounting, so the folder is created on Drive
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

---
## Pipeline

### A1. Install

PaddlePaddle ships CUDA-specific wheels from its own index. `cu126` is the build the PaddleOCR-VL docs show; Colab's driver is newer, which is fine (drivers are backward compatible). If install or the check below fails, try `PADDLE_VER = '3.3.1'` or the `cu129` index.

**Speed-ups kept on Drive** (`thesis/wheels/`, `thesis/paddlex_models/`): the first session downloads the Paddle wheel (~2 GB) and the OCR models (~2 GB) and saves them to Drive; later sessions copy them from Drive, which is much faster. Delete those folders to force a fresh download (e.g. after changing `PADDLE_VER`).


In [ ]:
PADDLE_VER = '3.2.1'
# Paddle's download server is slow from Colab (~2 GB wheel). The wheel is kept on Drive after the
# first session, so later sessions install it from Drive instead of downloading it again.
import os, glob
WHEELS = f'{THESIS}/wheels/paddlepaddle-gpu-{PADDLE_VER}-cu126'
os.makedirs(WHEELS, exist_ok=True)
if not glob.glob(f'{WHEELS}/paddlepaddle_gpu-*.whl'):
    print('first time: downloading the Paddle wheel to Drive (slow, once)')
    !pip download -q --no-deps --timeout 600 --retries 10 -d "{WHEELS}" paddlepaddle-gpu=={PADDLE_VER} -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
else:
    print('Paddle wheel found on Drive:', os.path.basename(glob.glob(f'{WHEELS}/paddlepaddle_gpu-*.whl')[0]))
!pip install -q --find-links "{WHEELS}" paddlepaddle-gpu=={PADDLE_VER}
!pip install -q -U "paddleocr[doc-parser]" pymupdf

# paddlepaddle-gpu pulls in an older NCCL that breaks Colab's torch
# ('undefined symbol: ncclCommWindowDeregister'), and paddleocr imports torch if present.
# Put back the NCCL version torch was built for. Newer NCCL stays compatible with paddle.
from importlib.metadata import requires
nccl = [r.split(';')[0].strip() for r in requires('torch') if r.startswith('nvidia-nccl')]
print('restoring', nccl)
!pip install -q --force-reinstall --no-deps "{nccl[0]}"

# Stop here if the install did not finish: a failed download does not stop the cell.
!pip show paddlepaddle-gpu paddleocr | grep -E '^(Name|Version)'


**Restart the session now** (Runtime → Restart session), then run the Settings and mount cells again and continue from the check below. Do not reinstall.

The install replaces packages Colab already loaded (numpy, protobuf, ...). Importing `paddleocr` without a restart fails, and any second attempt then fails with `PDX has already been initialized`, which hides the real error. If `from paddleocr import PaddleOCRVL` fails after a clean restart, the first error message is the one that matters.

In [ ]:
import paddle
paddle.utils.run_check()          # must end with 'PaddlePaddle is installed successfully!'
print(paddle.__version__, paddle.device.get_device())

### A2. Render pages

In [ ]:
import os, pymupdf
!rm -rf /content/pages /content/paddle_raw     # the previous paper's pages must not mix in
os.makedirs('/content/pages', exist_ok=True)
pdf = pymupdf.open(PDF)
pages = PAGES if PAGES is not None else range(pdf.page_count)
images = []
for p in pages:
    path = f'/content/pages/p{p}.png'
    pdf[p].get_pixmap(dpi=DPI).save(path)
    images.append((p, path))
print(images)

### A3. Run

First run downloads the layout and VL models (~2 GB). If it runs out of GPU memory: lower `DPI` to 120, or restart and run one page at a time.

In [ ]:
from paddleocr import PaddleOCRVL
# The OCR models (~2 GB) are downloaded to ~/.paddlex the first time. They are kept on Drive and
# copied back in later sessions. If a path differs in your PaddleOCR version, this only skips the copy.
import os, shutil
LOCAL_MODELS, DRIVE_MODELS = os.path.expanduser('~/.paddlex'), f'{THESIS}/paddlex_models'
if os.path.isdir(DRIVE_MODELS) and 'pipeline' not in globals():
    print('copying the OCR models from Drive ...')      # the import may already have created ~/.paddlex
    shutil.copytree(DRIVE_MODELS, LOCAL_MODELS, dirs_exist_ok=True)

if 'pipeline' not in globals():       # loaded once per session; the next paper reuses it
    pipeline = PaddleOCRVL()          # default pipeline version (v1.6 per the docs), PaddlePaddle engine

os.makedirs('/content/paddle_raw', exist_ok=True)
for p, path in images:
    for res in pipeline.predict(path):
        res.save_to_json(save_path=f'/content/paddle_raw/p{p}')
        res.save_to_markdown(save_path=f'/content/paddle_raw/p{p}')
!find /content/paddle_raw -type f

if os.path.isdir(LOCAL_MODELS) and not os.path.isdir(DRIVE_MODELS):
    print('saving the OCR models to Drive for next time ...')
    shutil.copytree(LOCAL_MODELS, DRIVE_MODELS)


### A4. Collect formulas → `paddle.json`

The label names for formulas aren't fully listed in the docs, so this prints every label it sees and keeps any label containing `formula`. Check the printed counts: if equations show up under another label, add it to `KEEP`.

Equation numbers: taken from a `\tag{n}` or trailing `(n)` in the LaTeX if present, otherwise from a number block on the same line.

In [ ]:
import json, glob, re, collections

def load_blocks(p):
    f = glob.glob(f'/content/paddle_raw/p{p}/**/*.json', recursive=True)[0]
    return json.load(open(f))['parsing_res_list']

def eq_number(text):
    m = re.search(r'\\tag\{(\d+)\}', text) or re.search(r'\((\d+)\)\s*\$*\s*$', text)
    return int(m.group(1)) if m else None

def y_overlap(a, b):
    return min(a[3], b[3]) - max(a[1], b[1]) > 0.5 * min(a[3] - a[1], b[3] - b[1])

labels = collections.Counter()
out = []
for p, _ in images:
    blocks = load_blocks(p)
    labels.update(b['block_label'] for b in blocks)
    numbers = [b for b in blocks if 'number' in b['block_label']]
    for k, b in enumerate(blocks):
        lab = b['block_label']
        if 'formula' not in lab or 'number' in lab or 'inline' in lab:   # inline math inside a sentence is not an equation
            continue
        tex, box = b['block_content'], list(map(float, b['block_bbox']))
        n = eq_number(tex)
        if n is None:
            # the number belongs to THIS equation only if it sits on the same line AND to its right.
            # Same line alone is not enough on a two-column page: the left column's (2) sits at the
            # same height as the right column's (21).
            near = []
            for x in numbers:
                xb = list(map(float, x['block_bbox']))
                gap = xb[0] - box[2]                       # number's left edge minus formula's right edge
                if y_overlap(box, xb) and gap > -10:
                    near.append((gap, x))
            near.sort(key=lambda t: t[0])
            if not near and k + 1 < len(blocks) and 'number' in blocks[k + 1]['block_label']:
                near = [(0, blocks[k + 1])]               # fallback: the number block that follows in reading order
            m = re.search(r'\d+', near[0][1]['block_content']) if near else None
            n = int(m.group()) if m else None
        out.append({'eq': n, 'page': p, 'label': lab, 'latex': tex, 'bbox': box, 'route': 'pipeline'})

print('labels seen:', dict(labels), '\n')
nums = [r['eq'] for r in out if r['eq'] is not None]
bad = [(a, b) for a, b in zip(nums, nums[1:]) if b <= a]
print('numbering check:', 'OK, increasing in reading order' if not bad else f'NOT increasing at {bad}: check these by hand', '\n')
for r in out:
    print(f"{str(r['eq']):>4}  {r['latex'][:150]}")
print(f'\n{len(out)} formulas, {sum(r["eq"] is None for r in out)} without a number')

json.dump({'paper': PAPER_ID, 'reader': 'PaddleOCR-VL pipeline', 'dpi': DPI, 'formulas': out},
          open(f'{OUT_DIR}/paddle.json', 'w'), indent=1, ensure_ascii=False)
print('saved ->', f'{OUT_DIR}/paddle.json')

### A5. Collect all blocks → `paddle_blocks.json`

Every block in reading order: text, titles, captions, tables (HTML), formulas. The extractor cuts these into sentences, list lines and table rows.


In [ ]:
rows = []
for p, _ in images:
    for k, b in enumerate(load_blocks(p)):
        rows.append({'page': p, 'order': k, 'label': b['block_label'], 'content': b['block_content'],
                     'bbox': list(map(float, b['block_bbox']))})

print(collections.Counter(r['label'] for r in rows))
json.dump({'paper': PAPER_ID, 'reader': 'PaddleOCR-VL pipeline', 'dpi': DPI, 'blocks': rows},
          open(f'{OUT_DIR}/paddle_blocks.json', 'w'), indent=1, ensure_ascii=False)
print(f'{len(rows)} blocks saved ->', f'{OUT_DIR}/paddle_blocks.json')


---
## B. Supplementary material (optional)

Only when part of the model is in the supplementary file. Upload it as `thesis/papers/<PAPER_ID>_supp.pdf`, run A2 to A5 for the main paper first, then this cell. It OCRs only the supplementary and appends its equations and text to the same two JSON files, tagged `source: supplementary`. Running it again replaces the supplementary part.


In [ ]:
import os, json, glob, re, collections, pymupdf

SUPP_PDF = f'{THESIS}/papers/{PAPER_ID}_supp.pdf'
assert os.path.exists(SUPP_PDF), f'not found: {SUPP_PDF}'
assert os.path.exists(f'{OUT_DIR}/paddle.json'), 'run A2-A5 for the main paper first'

!rm -rf /content/supp_pages /content/supp_raw
os.makedirs('/content/supp_pages', exist_ok=True)
doc = pymupdf.open(SUPP_PDF)
supp_images = []
for p in range(doc.page_count):
    path = f'/content/supp_pages/s{p}.png'
    doc[p].get_pixmap(dpi=DPI).save(path)
    supp_images.append((p, path))

from paddleocr import PaddleOCRVL
if 'pipeline' not in globals():
    pipeline = PaddleOCRVL()
for p, path in supp_images:
    for res in pipeline.predict(path):
        res.save_to_json(save_path=f'/content/supp_raw/s{p}')

def supp_blocks(p):
    return json.load(open(glob.glob(f'/content/supp_raw/s{p}/**/*.json', recursive=True)[0]))['parsing_res_list']

def supp_number(text):        # (A-3), (S1), (12a): kept as text
    m = re.search(r'\\tag\{([^}]+)\}', text) or re.search(r'\(([A-Z]{0,2}\s*[-.]?\s*\d+[a-z]?)\)\s*\$*\s*$', text)
    return re.sub(r'\s', '', m.group(1)) if m else None

def y_overlap(a, b):
    return min(a[3], b[3]) - max(a[1], b[1]) > 0.5 * min(a[3] - a[1], b[3] - b[1])

formulas, rows = [], []
for p, _ in supp_images:
    blocks = supp_blocks(p)
    numbers = [b for b in blocks if 'number' in b['block_label']]
    for k, b in enumerate(blocks):
        box, lab = list(map(float, b['block_bbox'])), b['block_label']
        rows.append({'page': p, 'order': k, 'label': lab, 'content': b['block_content'], 'bbox': box, 'source': 'supplementary'})
        if 'formula' not in lab or 'number' in lab or 'inline' in lab:
            continue
        n = supp_number(b['block_content'])
        if n is None:
            near = sorted(((list(map(float, x['block_bbox']))[0] - box[2], x) for x in numbers
                           if y_overlap(box, list(map(float, x['block_bbox']))) and list(map(float, x['block_bbox']))[0] - box[2] > -10),
                          key=lambda t: t[0])
            if not near and k + 1 < len(blocks) and 'number' in blocks[k + 1]['block_label']:
                near = [(0, blocks[k + 1])]
            n = supp_number(near[0][1]['block_content']) if near else None
        formulas.append({'eq': n, 'page': p, 'label': lab, 'latex': b['block_content'], 'bbox': box,
                         'route': 'pipeline', 'source': 'supplementary'})

eq_file, bl_file = f'{OUT_DIR}/paddle.json', f'{OUT_DIR}/paddle_blocks.json'
E, B = json.load(open(eq_file)), json.load(open(bl_file))
E['formulas'] = [{**f, 'source': f.get('source', 'main')} for f in E['formulas'] if f.get('source') != 'supplementary'] + formulas
B['blocks'] = [{**b, 'source': b.get('source', 'main')} for b in B['blocks'] if b.get('source') != 'supplementary'] + rows
json.dump(E, open(eq_file, 'w'), indent=1, ensure_ascii=False)
json.dump(B, open(bl_file, 'w'), indent=1, ensure_ascii=False)
for f in formulas:
    print(f"{str(f['eq']):>6}  p.{f['page'] + 1}  {f['latex'][:120]}")
print(f"\n{len(formulas)} supplementary formulas, {sum(f['eq'] is None for f in formulas)} without a number -> {eq_file}")
